# Customer Support Ticket Intelligence — Google Colab

This notebook runs the TF-IDF baseline, fine-tunes DistilBERT, builds semantic retrieval, and evaluates both classifiers.

Before running: choose **Runtime → Change runtime type → T4 GPU** (or another available GPU), then run cells top to bottom. Colab hardware and usage limits vary. The dataset is synthetic, so results describe this dataset only.

In [ ]:
from pathlib import Path
import os, subprocess
REPO_URL = 'https://github.com/Indrajeetkumar-kgp/Customer-Support-Ticket-Intelligence.git'
PROJECT_DIR = Path('/content/Customer-Support-Ticket-Intelligence')
if not (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print('Project folder:', PROJECT_DIR)

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU attached. Select Runtime → Change runtime type → T4 GPU, reconnect, and rerun.')
print('GPU:', torch.cuda.get_device_name(0))

## 1. Train the baseline and transformer

Both models use the same duplicate-aware split with seed 42. DistilBERT starts from pretrained weights and fine-tunes on the full training split. The GPU settings below are a starting point; if you run out of memory, reduce batch size to 8.

In [ ]:
!python -m src.train_baseline --seed 42

In [ ]:
!python -m src.train_transformer --seed 42 --epochs 3 --batch-size 16 --max-length 64

## 2. Build retrieval and evaluate on the held-out test split

In [ ]:
!python -m src.retrieval --seed 42
!python -m src.evaluate --seed 42 --model both

In [ ]:
import json
import pandas as pd
from pathlib import Path
evaluation_dir = Path('artifacts/evaluation')
rows = []
for name in ['baseline', 'transformer']:
    report = json.loads((evaluation_dir / f'{name}_metrics.json').read_text())
    rows.append({'model': name, 'accuracy': report['accuracy'], 'macro_f1': report['macro_f1'], 'weighted_f1': report['weighted_f1']})
summary = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
display(summary.style.format({'accuracy': '{:.4f}', 'macro_f1': '{:.4f}', 'weighted_f1': '{:.4f}'}))
summary.to_csv(evaluation_dir / 'summary.csv', index=False)
print('Per-class metrics, predictions, and confusion matrices:', evaluation_dir)

## 3. Try a sample message and download results

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from src.retrieval import SemanticIndex
model_dir = Path('artifacts/transformer/model')
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir).to('cuda').eval()
index = SemanticIndex('artifacts/retrieval')
message = 'I need help cancelling an order.'
inputs = tokenizer(message, return_tensors='pt', truncation=True, max_length=64).to('cuda')
with torch.inference_mode():
    probabilities = model(**inputs).logits.softmax(dim=-1)[0]
predicted_id = int(probabilities.argmax())
print('Message:', message)
print('Predicted intent:', model.config.id2label[predicted_id])
print(f'Confidence: {float(probabilities[predicted_id]):.3f}')
display(index.search(message, top_k=5))

In [ ]:
from google.colab import files
files.download('artifacts/evaluation/summary.csv')

Save or download `artifacts/evaluation/` before the Colab runtime ends. Colab sessions are temporary. To review the results together, share the summary table and the per-class metrics from the evaluation JSON files.